In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval "maite-datasets[hf]>=0.0.25"
except Exception:
    pass

In [2]:
import warnings

import numpy as np
import polars as pl
from IPython.display import display
from maite_datasets.multiobject_tracking import VisDroneMOT

from dataeval import Metadata
from dataeval.config import set_max_processes
from dataeval.core import image_stats
from dataeval.data import EvenlySpaced, Indices, SequenceFrames, View
from dataeval.flags import ImageStats
from dataeval.utils.preprocessing import ChannelGroup

set_max_processes(4)
# Reading rows bins the numeric factors, which this guide does not use.
warnings.filterwarnings("ignore", message=".*binned automatically")

In [3]:
dataset = VisDroneMOT(root="./data", image_set="val", download=True)
videos = Metadata(dataset).rows_at("sequence")
day = videos.filter((pl.col("time_of_day") == "daytime") & (pl.col("scene_type") != "sporting event"))
night = videos.filter(pl.col("time_of_day") == "night")

collections = {
    "reference": SequenceFrames(View(dataset, Indices(day["item_index"].to_list())), EvenlySpaced(15)),
    "operational": SequenceFrames(View(dataset, Indices(night["item_index"].to_list())), EvenlySpaced(60)),
}
for name, frames in collections.items():
    print(f"{name:12} {len(frames)} frames")

reference    60 frames
operational  60 frames


In [4]:
settings = {
    "stats": ImageStats.CONTEXT | ImageStats.PIXEL_SATURATED_FRACTION | ImageStats.NOISE,
    "value_range": (0, 255),
}
results = {name: image_stats(frames, **settings) for name, frames in collections.items()}
reference, operational = results["reference"], results["operational"]

print("settings match:", reference.get("measurement_specs", {}) == operational.get("measurement_specs", {}))
print(reference.get("measurement_specs", {})["target_surround_contrast"])

settings match: True
MeasurementSpec(algorithm='target_surround_contrast', version='1', units='ratio', region='target and surround', normalization='[0, 1]', config=mappingproxy({'padding': 0.25, 'min_pixels': 32, 'min_target_valid_fraction': 0.9, 'intensity_floor': 0.00392156862745098}), range_policy='declared', declared_range=(0.0, 255.0), rgb_channels=None)


In [5]:
metadata = {}
for name, frames in collections.items():
    metadata[name] = Metadata(frames)
    metadata[name].add_factors(results[name])

vehicles = [i for i, label in dataset.index2label.items() if label in ("car", "van", "truck", "bus")]
boxes = pl.concat(
    metadata[name]
    .rows_at("instance")
    .filter(pl.col("class_label").is_in(vehicles))
    .select(
        pl.lit(name).alias("collection"),
        pl.col("instance_target_surround_contrast").fill_nan(None).alias("contrast"),
        pl.col("instance_signal_clutter_ratio").fill_nan(None).alias("scr"),
    )
    for name in collections
)

display(
    boxes.group_by("collection", maintain_order=True).agg(
        pl.len().alias("vehicle boxes"),
        pl.col("contrast").is_not_null().mean().round(4).alias("share with a value"),
        pl.col("contrast").median().round(3).alias("median contrast"),
        pl.col("scr").median().round(3).alias("median scr"),
    )
)

collection,vehicle boxes,share with a value,median contrast,median scr
str,u32,f64,f32,f32
"""reference""",1347,0.9993,0.034,0.167
"""operational""",939,1.0,0.122,0.337


In [6]:
edges = {
    "contrast": [-np.inf, -0.4, -0.2, 0.0, 0.2, 0.4, np.inf],
    "scr": [-np.inf, -2.0, -1.0, 0.0, 1.0, 2.0, np.inf],
}
tables = {}
for column, cuts in edges.items():
    shares = {}
    for name in collections:
        values = boxes.filter(pl.col("collection") == name)[column].drop_nulls().to_numpy()
        shares[name] = np.round(np.histogram(values, cuts)[0] / len(values), 3)
    bins = [f"{low} to {high}" for low, high in zip(cuts[:-1], cuts[1:], strict=True)]
    tables[column] = pl.DataFrame({f"{column} bin": bins, **shares})
    display(tables[column])

contrast bin,reference,operational
str,f64,f64
"""-inf to -0.4""",0.005,0.001
"""-0.4 to -0.2""",0.148,0.072
"""-0.2 to 0.0""",0.285,0.262
"""0.0 to 0.2""",0.322,0.305
"""0.2 to 0.4""",0.183,0.183
"""0.4 to inf""",0.057,0.177


scr bin,reference,operational
str,f64,f64
"""-inf to -2.0""",0.059,0.005
"""-2.0 to -1.0""",0.112,0.02
"""-1.0 to 0.0""",0.267,0.31
"""0.0 to 1.0""",0.338,0.414
"""1.0 to 2.0""",0.127,0.19
"""2.0 to inf""",0.097,0.061


In [7]:
per_video = pl.concat(
    metadata[name]
    .rows_at("unit")
    .group_by("source_id")
    .agg(
        pl.len().alias("frames"),
        (pl.col("unit_saturated_fraction") > 0.01).mean().alias("share over 1% clipped"),
        pl.col("unit_saturated_fraction").median().round(4).alias("median clipped"),
        pl.col("unit_noise_std").median().round(2).alias("median noise_std"),
    )
    .sort("source_id")
    .select(pl.lit(name).alias("collection"), pl.all())
    for name in collections
)
display(per_video)
for name, result in results.items():
    print(f"{name:12} clipping endpoint: {np.unique(np.asarray(result.get('details', {})['saturated_upper']))}")

collection,source_id,frames,share over 1% clipped,median clipped,median noise_std
str,str,u32,f64,f32,f32
"""reference""","""uav0000137_00458_v""",15,0.0,0.0002,0.91
"""reference""","""uav0000182_00000_v""",15,0.0,0.0002,2.49
"""reference""","""uav0000268_05773_v""",15,1.0,0.2987,1.31
"""reference""","""uav0000305_00000_v""",15,0.0,0.0006,0.87
"""operational""","""uav0000117_02622_v""",60,0.0,0.0009,0.56


reference    clipping endpoint: [255.]
operational  clipping endpoint: [255.]


In [8]:
night_frames = [collections["operational"][i] for i in range(len(collections["operational"]))]
twelve_bit = [np.round(np.asarray(image, np.float64) * 4095 / 255).astype(np.uint16) for image, _, _ in night_frames]
twelve_bit_boxes = [np.asarray(target.boxes) for _, target, _ in night_frames]
ratios_and_clipping = (
    ImageStats.CONTEXT_TARGET_SURROUND_CONTRAST
    | ImageStats.CONTEXT_SIGNAL_CLUTTER_RATIO
    | ImageStats.PIXEL_SATURATED_FRACTION
)

undeclared = image_stats(twelve_bit, boxes=twelve_bit_boxes, stats=ratios_and_clipping)
print("context statuses:", sorted(set(np.asarray(undeclared.get("details", {})["context_status"]).tolist())))
print("clipping endpoint:", np.unique(np.asarray(undeclared.get("details", {})["saturated_upper"])))

context statuses: ['not_applicable', 'unknown_range']
clipping endpoint: [nan]


In [9]:
declared = image_stats(
    twelve_bit,
    boxes=twelve_bit_boxes,
    stats={"rgb": ratios_and_clipping},
    channels={"rgb": ChannelGroup([0, 1, 2], value_range=(0, 4095))},
)
eight_bit = operational["stats"]
contrast_gap = np.nanmax(
    np.abs(np.asarray(declared["stats"]["rgb_target_surround_contrast"]) - eight_bit["target_surround_contrast"])
)
print(f"largest contrast difference from the 8-bit frames: {contrast_gap:.4f}")
print(
    "same clipped shares:",
    np.array_equal(declared["stats"]["rgb_saturated_fraction"], eight_bit["saturated_fraction"], equal_nan=True),
)
print("clipping endpoint:", np.unique(np.asarray(declared.get("details", {})["rgb_saturated_upper"])))
print("declared range:", declared.get("measurement_specs", {})["rgb_saturated_fraction"].declared_range)

largest contrast difference from the 8-bit frames: 0.0023
same clipped shares: True
clipping endpoint: [4095.]
declared range: (0.0, 4095.0)


In [10]:
# TEST ASSERTION CELL ###
assert {name: len(frames) for name, frames in collections.items()} == {"reference": 60, "operational": 60}
assert reference.get("measurement_specs", {}) == operational.get("measurement_specs", {})
assert "noise_std" not in reference.get("measurement_specs", {})
inferred = image_stats(collections["operational"], stats=ImageStats.PIXEL_SATURATED_FRACTION)
assert (
    inferred.get("measurement_specs", {})["saturated_fraction"]
    != operational.get("measurement_specs", {})["saturated_fraction"]
)
assert np.array_equal(inferred["stats"]["saturated_fraction"], eight_bit["saturated_fraction"], equal_nan=True)

summary = {
    row["collection"]: row
    for row in boxes
    .group_by("collection")
    .agg(
        pl.len().alias("n"),
        pl.col("contrast").null_count().alias("missing"),
        pl.col("contrast").median(),
        pl.col("scr").median(),
    )
    .iter_rows(named=True)
}
assert summary["reference"]["n"] == 1347
assert summary["operational"]["n"] == 939
assert summary["reference"]["missing"] == 1
assert summary["operational"]["missing"] == 0
assert 0.02 < summary["reference"]["contrast"] < 0.05
assert 0.10 < summary["operational"]["contrast"] < 0.15
assert 0.14 < summary["reference"]["scr"] < 0.20
assert 0.30 < summary["operational"]["scr"] < 0.38

contrast_shares, scr_shares = tables["contrast"], tables["scr"]
assert 0.14 < contrast_shares["operational"][-1] < 0.22
assert 0.03 < contrast_shares["reference"][-1] < 0.09
day_wide = scr_shares["reference"][0] + scr_shares["reference"][-1]
night_wide = scr_shares["operational"][0] + scr_shares["operational"][-1]
assert 0.12 < day_wide < 0.20
assert 0.04 < night_wide < 0.10

clipped = per_video.filter(pl.col("share over 1% clipped") > 0)
assert clipped["collection"].to_list() == ["reference"]
assert clipped["share over 1% clipped"].to_list() == [1.0]
frame_rows = pl.concat(
    metadata[name].rows_at("unit").select("source_id", "unit_saturated_fraction") for name in collections
)
assert frame_rows.filter(pl.col("source_id") != clipped["source_id"][0])["unit_saturated_fraction"].max() < 0.001
for result in results.values():
    assert np.unique(np.asarray(result.get("details", {})["saturated_upper"])).tolist() == [255]
day_noise = per_video.filter(pl.col("collection") == "reference")["median noise_std"]
night_noise = per_video.filter(pl.col("collection") == "operational")["median noise_std"][0]
assert day_noise.max() - day_noise.min() > day_noise.min() - night_noise > 0

statuses = np.asarray(undeclared.get("details", {})["context_status"])
assert set(statuses.tolist()) == {"not_applicable", "unknown_range"}
assert (statuses == "unknown_range").sum() == sum(len(b) for b in twelve_bit_boxes)
assert np.isnan(np.asarray(undeclared["stats"]["target_surround_contrast"])).all()
assert np.isnan(np.asarray(undeclared.get("details", {})["saturated_upper"])).all()
assert contrast_gap < 0.005
assert np.array_equal(declared["stats"]["rgb_saturated_fraction"], eight_bit["saturated_fraction"], equal_nan=True)
assert np.unique(np.asarray(declared.get("details", {})["rgb_saturated_upper"])).tolist() == [4095]
assert declared.get("measurement_specs", {})["rgb_saturated_fraction"].declared_range == (0, 4095)
assert (
    declared.get("measurement_specs", {})["rgb_saturated_fraction"]
    != operational.get("measurement_specs", {})["saturated_fraction"]
)